In [5]:
import os
import numpy as np
import librosa
import soundfile as sf

# Optional: scipy for filters
try:
    from scipy.signal import butter, filtfilt, iirnotch
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("[WARN] scipy not available, filter-based cleaning will be skipped:", e)

AUDIO_PATH = "/home/jsudan/audio_anaysis/Jinhan-gil TV News Shocking Recording File... People Appearing in the Audio (Park Seon-won, Gwa).wav"
TARGET_SR = 16000
OUT_DIR = "/home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio"
os.makedirs(OUT_DIR, exist_ok=True)


In [6]:
y, sr = librosa.load(AUDIO_PATH, sr=TARGET_SR, mono=True)
print("Loaded", AUDIO_PATH)
print("sr=", sr, "seconds=", len(y)/sr)

Loaded /home/jsudan/audio_anaysis/Jinhan-gil TV News Shocking Recording File... People Appearing in the Audio (Park Seon-won, Gwa).wav
sr= 16000 seconds= 400.2191875


In [7]:
# ------------------ helpers ------------------

def normalize_peak(x, peak=0.99):
    m = np.max(np.abs(x))
    if m == 0:
        return x
    return x * (peak / m)


def butter_filter(x, sr, cutoff, btype, order=4):
    if not SCIPY_OK:
        return x
    nyq = 0.5 * sr
    norm = np.array(cutoff, dtype=float) / nyq
    # Keep critical frequencies strictly inside (0, 1)
    norm = np.clip(norm, 1e-6, 0.999)
    b, a = butter(order, norm, btype=btype)
    return filtfilt(b, a, x)


def notch_filter(x, sr, freq=60.0, q=30.0):
    if not SCIPY_OK:
        return x
    b, a = iirnotch(freq, q, fs=sr)
    return filtfilt(b, a, x)


def spectral_gate(x, sr, noise_sec=0.5, reduce_factor=1.5):
    # Simple spectral subtraction using initial noise profile
    n_noise = int(noise_sec * sr)
    noise_clip = x[:n_noise] if len(x) > n_noise else x
    stft = librosa.stft(x, n_fft=1024, hop_length=256)
    mag, phase = np.abs(stft), np.exp(1j * np.angle(stft))

    noise_stft = librosa.stft(noise_clip, n_fft=1024, hop_length=256)
    noise_mag = np.mean(np.abs(noise_stft), axis=1, keepdims=True)

    clean_mag = np.maximum(mag - noise_mag * reduce_factor, 0.0)
    clean_stft = clean_mag * phase
    y_clean = librosa.istft(clean_stft, hop_length=256, length=len(x))
    return y_clean

In [8]:
# ------------------ clean variants ------------------
clean = {}

# 0) raw (normalized only)
clean["raw_norm"] = normalize_peak(y)

# 1) gentle high-pass (remove rumble)
clean["highpass_80hz"] = normalize_peak(butter_filter(y, sr, cutoff=80, btype="highpass"))

# 2) gentle low-pass (remove hiss)
clean["lowpass_8000hz"] = normalize_peak(butter_filter(y, sr, cutoff=8000, btype="lowpass"))

# 3) band-pass (speech band)
clean["bandpass_80_8000hz"] = normalize_peak(butter_filter(y, sr, cutoff=[80, 8000], btype="bandpass"))

# 4) notch 60Hz (hum)
clean["notch_60hz"] = normalize_peak(notch_filter(y, sr, freq=60.0))

# 5) light spectral gate (steady noise)
clean["spectral_gate"] = normalize_peak(spectral_gate(y, sr, noise_sec=0.5, reduce_factor=1.2))

In [9]:
# ------------------ save outputs ------------------
for name, audio in clean.items():
    out_path = os.path.join(OUT_DIR, f"{name}.wav")
    sf.write(out_path, audio, sr)
    print("wrote", out_path)

wrote /home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio/raw_norm.wav
wrote /home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio/highpass_80hz.wav
wrote /home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio/lowpass_8000hz.wav
wrote /home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio/bandpass_80_8000hz.wav
wrote /home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio/notch_60hz.wav
wrote /home/jsudan/wav2vec_contr_loss/one_audio_analysis/clean_audio/spectral_gate.wav
